# EDA: Pokémon Type Prediction Dataset

Análisis exploratorio sencillo del dataset de Pokémon para entender la distribución de tipos y características especiales.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
PROJECT_ROOT = Path.cwd().parent
pokemon_path = PROJECT_ROOT / 'data' / 'raw' / 'pokemon.json'

with open(pokemon_path, 'r') as f:
    pokemon_data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(pokemon_data)

print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head(10))
print(f"\nData types:")
print(df.dtypes)
print(f"\nBasic stats:")
print(df.info())

## 1. Distribución de Tipos (Type1)

In [ ]:
# Distribution of primary types
type1_counts = df['type1'].value_counts().sort_values(ascending=False)
print(f"Primary Type Distribution (total {len(type1_counts)} unique types):")
print(type1_counts)
print(f"\nTotal Pokémon: {len(df)}")

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
type1_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Count')
ax.set_title('Primary Type Distribution')
plt.tight_layout()
plt.show()

## 2. Análisis de Pokémon Voladores (Flying Type)

In [ ]:
# Flying type analysis
flying_primary = df[df['type1'] == 'flying']
flying_secondary = df[df['type2'] == 'flying']
flying_pure = flying_primary[flying_primary['type2'].isna()]
flying_secondary_only = flying_secondary[flying_secondary['type1'] != 'flying']

print(f"Pokémon Voladores:\n")
print(f"  - Tipo primario volador (type1='flying'): {len(flying_primary)}")
print(f"    - Puros (solo volador): {len(flying_pure)}")
print(f"  - Tipo secundario volador (type2='flying'): {len(flying_secondary)}")
print(f"    - Solo secundario: {len(flying_secondary_only)}")
print(f"\nTotal con tipo volador: {len(flying_primary) + len(flying_secondary_only)}")

# Show the pure flying types
print(f"\nPokémon voladores PUROS (type1=flying, type2=null):")
print(flying_pure[['name', 'id', 'type1', 'type2']].to_string())

In [ ]:
# Flying with primary type
print(f"Pokémon con tipo primario FLYING ({len(flying_primary)}):")
print(flying_primary[['name', 'id', 'type1', 'type2']].to_string())

print(f"\n" + "="*60)
print(f"\nTipos secundarios de Pokémon voladores primarios:")
print(flying_primary['type2'].value_counts(dropna=False))

## 3. Análisis de Balance de Clases

In [ ]:
# Class balance analysis
type_dist = df['type1'].value_counts()

print(f"Estadísticas de distribución de clases (type1):")
print(f"  - Media: {type_dist.mean():.2f}")
print(f"  - Desv Est: {type_dist.std():.2f}")
print(f"  - Min: {type_dist.min()} ({type_dist.idxmin()})")
print(f"  - Max: {type_dist.max()} ({type_dist.idxmax()})")
print(f"  - Mediana: {type_dist.median():.2f}")

# Class imbalance ratio
print(f"\nRatio des-balance (Max/Min): {type_dist.max() / type_dist.min():.2f}x")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of class distribution
axes[0].hist(type_dist.values, bins=15, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Pokémon per Type')
axes[0].set_ylabel('Number of Types')
axes[0].set_title('Distribution of Class Sizes')
axes[0].axvline(type_dist.mean(), color='red', linestyle='--', label=f'Mean: {type_dist.mean():.1f}')
axes[0].legend()

# Box plot
axes[1].boxplot(type_dist.values)
axes[1].set_ylabel('Pokémon per Type')
axes[1].set_title('Box Plot of Class Sizes')
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

## 4. Tipos Mono vs Dual

In [ ]:
# Mono-type vs Dual-type
monotype = df[df['type2'].isna()]
dualtype = df[df['type2'].notna()]

print(f"Tipos puros (mono-type): {len(monotype)} ({100*len(monotype)/len(df):.1f}%)")
print(f"Tipos duales (dual-type): {len(dualtype)} ({100*len(dualtype)/len(df):.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
sizes = [len(monotype), len(dualtype)]
labels = [f'Mono-type\n({len(monotype)})', f'Dual-type\n({len(dualtype)})']
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=['lightblue', 'lightcoral'])
axes[0].set_title('Mono-type vs Dual-type Distribution')

# Distribution of secondary types
type2_counts = df[df['type2'].notna()]['type2'].value_counts()
type2_counts.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Count')
axes[1].set_title('Secondary Type Distribution (for dual-type Pokémon)')

plt.tight_layout()
plt.show()

## 5. Resumen de Desafíos para ML

In [ ]:
# Summary of challenges
print("""\n╔════════════════════════════════════════════════════════════════╗
║  RESUMEN DE DESAFÍOS PARA ML                                   ║
╚════════════════════════════════════════════════════════════════╝

1. DESBALANCE DE CLASES:\n""")
print(f"   - Tipo más frecuente: {type_dist.idxmax()} ({type_dist.max()} Pokémon)")
print(f"   - Tipo menos frecuente: {type_dist.idxmin()} ({type_dist.min()} Pokémon)")
print(f"   - Ratio: {type_dist.max() / type_dist.min():.1f}x")

print(f"\n2. TIPOS RAROS (< 5 Pokémon por tipo):")
rare_types = type_dist[type_dist < 5]
print(f"   - Cantidad: {len(rare_types)} tipos")
for t, count in rare_types.items():
    print(f"     - {t}: {count}")

print(f"\n3. POKÉMON VOLADORES (Flying type):")
print(f"   - Voladores primarios: {len(flying_primary)}")
print(f"   - Voladores puros: {len(flying_pure)}")
print(f"   - Total con tipo volador: {len(flying_primary) + len(flying_secondary_only)}")
print(f"   ⚠️  Muy pocos para CV estratificada")

print(f"\n4. CONFUSION POTENCIAL (secundarios como primarios):")
confusion_potential = 0
for idx, row in df.iterrows():
    type2 = row['type2']
    if type2 is not None and type2 in df['type1'].values:
        confusion_potential += 1
print(f"   - Pokémon cuyo tipo2 es tipo1 de otros: {confusion_potential}")
print(f"   - Por qué: Proporciona insight XAI natural cuando el modelo predice tipo2\n")

## 6. Propuestas de Estrategia

In [ ]:
print("""\n╔════════════════════════════════════════════════════════════════╗
║  OPCIONES RECOMENDADAS PARA PROCEDER                          ║
╚════════════════════════════════════════════════════════════════╝

OPCIÓN A: Subset de tipos frecuentes (más fácil)
─────────────────────────────────────────────
• Seleccionar top 5-10 tipos con ≥ 50 Pokémon cada uno
• Proporciona balance razonable para CV y XAI
• Focus en interpretabilidad

OPCIÓN B: Usar todos los 18 tipos (desafiante pero interesante)
──────────────────────────────────────────────────────────────
• Stratified K-Fold con adjustment para clases pequeñas
• Técnicas de balanceo (oversampling/undersampling)
• El desbalance es realista y demuestra ML real

OPCIÓN C: Dividir por tipos de cartas/rareza (tu sugerencia)
──────────────────────────────────────────────────────────────
• Agrupar tipos en categorías temáticas
• Voladores, Físicos, Especiales, etc.
• Reduce clases y permite CV más limpia

RECOMENDACIÓN PARA VOLADORES:
───────────────────────────────
• Solo 9 voladores es muy poco para validación confiable
• Estrategia actual (mezclarlos en normal/secundario) es pragmática
• O combinar con otros 'raros' para formar una clase 'Other/Rare'
""")

## 7. Estadísticas Detalladas

In [ ]:
# Detailed statistics table
stats_df = pd.DataFrame({
    'Type': type_dist.index,
    'Count': type_dist.values,
    'Percentage': (100 * type_dist.values / len(df)).round(2),
    'Log2(Count)': np.log2(type_dist.values).round(2)
})

print("\nDetailed Type Distribution:")
print(stats_df.to_string(index=False))

# Export to CSV for later use
output_path = Path.cwd().parent / 'data' / 'processed' / 'type_distribution.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
stats_df.to_csv(output_path, index=False)
print(f"\n✓ Exported to: {output_path}")

In [ ]:
# Create a correlation matrix for type combinations
type_combo_matrix = pd.crosstab(df['type1'], df['type2'], margins=True)
print("\nType Combination Matrix (type1 vs type2):")
print(type_combo_matrix.iloc[:10, :10])  # Show top 10x10
print("... (truncated)")